# Preprocessing

## Análisis y preprocesado de los datos

Iniciamos leyendo los datos, que contienen varios errores a arreglar durante el preprocesado

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import numpy as np
import pandas as pd
from functions_scripts.load import read_data
from classes.Preparador import Preparador
from classes.Preparador2 import Preparador2
from classes.OutlierDetector import OutlierDetector

pd.set_option('future.no_silent_downcasting', True)

df = read_data(filepath="../data/Autism-Adult-Data.csv")

preparador = Preparador(df=df)
# preparador2 = Preparador2().fit(df)

     gender  jaundice  family_pdd  used_app_before  class   age  \
0         0         0           0                0      0  26.0   
1         1         0           1                0      0  24.0   
2         1         1           1                0      1  27.0   
3         0         0           1                0      0  35.0   
4         0         0           0                0      0  40.0   
..      ...       ...         ...              ...    ...   ...   
699       0         0           0                0      1  25.0   
700       1         0           0                0      0  34.0   
701       0         0           0                0      1  24.0   
702       1         0           0                0      0  35.0   
703       0         0           0                0      1  26.0   

          ethnicity country_of_res relation  a1_score  a2_score  a3_score  \
0    white-european   unitedstates     self         1         1         1   
1            latino         brazil     se

#### Reemplazo de nombres erróneos de columnas
Los nombres de columnas 'jundice' y 'Austim' son incorrectos, ya que se refieren a 'jaundice' y 'family_pdd' respectivamente.

In [25]:
# Las columnas Jundice y Austim se encuentran en las posiciones 14 y 15

idxs = [14, 15]

print("ANTES:")
preparador.column_get_unique(idxs)

preparador.column_rename(idxs, ["jaundice", "family_pdd"])

print("\nDESPUES:")
preparador.column_get_unique(idxs)

ANTES:
Unique vals in jundice: ['no' 'yes']
Unique vals in austim: ['no' 'yes']

DESPUES:
Unique vals in jaundice: ['no' 'yes']
Unique vals in family_pdd: ['no' 'yes']


#### Error en representación de valores perdidos

Los valores perdidos en este dataset están representados como ?, y nos sirve que estén representados como NaN, ya que sino las columnas quedarán como dtype='object' y necesitamos que sean numéricas.

Demostración del error:

In [26]:
preparador.column_get_unique("age") # Debería ser numérica pero no lo es
preparador.column_get_unique("result") # Debería ser numérica (en este caso no hace falta arreglarla 
                            # porque no hay valores perdidos)

Unique vals in age: ['17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30'
 '31' '32' '33' '34' '35' '36' '37' '38' '383' '39' '40' '41' '42' '43'
 '44' '45' '46' '47' '48' '49' '50' '51' '52' '53' '54' '55' '56' '58'
 '59' '60' '61' '64' '?']
Unique vals in result: [ 0  1  2  3  4  5  6  7  8  9 10]


In [27]:
preparador.column_to_numeric("age")


Confirmación de que se realizó la operación correctamente:

In [28]:
preparador.column_get_unique("age") # Debería ser numérica, ahora lo es

Unique vals in age: [ 17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.  28.  29.  30.
  31.  32.  33.  34.  35.  36.  37.  38.  39.  40.  41.  42.  43.  44.
  45.  46.  47.  48.  49.  50.  51.  52.  53.  54.  55.  56.  58.  59.
  60.  61.  64. 383.  nan]


### Errores de nombrado y columna innecesaria

Las columnas ID y age_desc no aportan información, ya que ID contiene valores diferentes incrementales por cada uno de los datos, y age_desc contiene un único valor diferente: '18 and more'

In [29]:
preparador.column_get_unique("age_desc") # Debería ser numérica, ahora lo es

print(f"# of rows in dataframe: {df.shape[0]}")
print(f"# of unique vals in ID: {len(np.unique(df.id))}")

preparador.column_del(["id", "age_desc"])

Unique vals in age_desc: ['18 and more']
# of rows in dataframe: 704
# of unique vals in ID: 704


### Columnas listas para valores faltantes, outliers e inconsistencias

Vamos a hacer una breve muestra de como quedaron los datos

In [30]:
preparador.describe_data()



Columna: A1_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.721591
std        0.448535
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: A1_Score, dtype: float64


Columna: A2_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.453125
std        0.498152
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A2_Score, dtype: float64


Columna: A3_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.457386
std        0.498535
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A3_Score, dtype: float64


Columna: A4_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.495739
std        0.500337
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.00

### Categorico a numérico

Tenemos varias columnas binarias cuyos valores son {'no', 'yes'}, {'f', 'm'}, ...

Estas columnas las reemplazaremos por columnas enteras con estos reemplazamientos:

    no, f = 0
    yes, m = 1

In [31]:
idxs_categoric_binary_columns = [11, 13, 14, 16, 19]

print("ANTES:")
preparador.column_get_unique(idxs_categoric_binary_columns)
preparador.column_binary_categoric_to_numeric(idxs_categoric_binary_columns)

print("\nDESPUES:")
preparador.column_get_unique(idxs_categoric_binary_columns)

ANTES:
Unique vals in gender: ['f' 'm']
Unique vals in jaundice: ['no' 'yes']
Unique vals in family_pdd: ['no' 'yes']
Unique vals in used_app_before: ['no' 'yes']
Unique vals in Class/ASD: ['NO' 'YES']

DESPUES:
Unique vals in gender: [0 1]
Unique vals in jaundice: [0 1]
Unique vals in family_pdd: [0 1]
Unique vals in used_app_before: [0 1]
Unique vals in Class/ASD: [0 1]


> Solo a modo de ejemplo, tenemos programado este paso a paso en una sola función, mostrada en el siguiente bloque de código

In [32]:
df = read_data(filepath="../data/Autism-Adult-Data.csv")

preparador = Preparador().fit(df)

preparador.describe_data()



Columna: A1_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.721591
std        0.448535
min        0.000000
25%        0.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: A1_Score, dtype: float64


Columna: A2_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.453125
std        0.498152
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A2_Score, dtype: float64


Columna: A3_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.457386
std        0.498535
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.000000
Name: A3_Score, dtype: float64


Columna: A4_Score
DataType: int64
Unique Vals: [0 1]
Description:
count    704.000000
mean       0.495739
std        0.500337
min        0.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        1.00

#### Tratamiento de outliers

Ahora debemos tratar los outliers de las columnas numéricas

Como solo estamos demostrando el análisis, no haremos GridSearch para encontrar los mejores hiperparámetros. Esto lo dejamos para el programa principal.

Detectamos el siguiente outlier en la única columna numérica no binaria:

- Age: 383

Como la detección de outliers utilizada para este ejemplo es por IQR, se reemplazan por la mediana también las edades:

- 64
- 61
- 59
- 58

In [33]:
outliers = OutlierDetector()
# Parámetros por defecto:
#   k         = 1.5
#   deteccion = "iqr"
#   reemplazo = "mediana"

# Obtenemos el dataframe preparado previamente
df = preparador.df

# Mostramos el outlier en la columna age
preparador.describe_data(df.age)

# Tratamos el outlier
df = outliers.fit_transform(df)

# Mostramos que el outlier ya está corregido luego de tratarlo
preparador.describe_data(df.age)



Columna: age
DataType: float32
Unique Vals: [ 17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.  28.  29.  30.
  31.  32.  33.  34.  35.  36.  37.  38.  39.  40.  41.  42.  43.  44.
  45.  46.  47.  48.  49.  50.  51.  52.  53.  54.  55.  56.  58.  59.
  60.  61.  64. 383.  nan]
Description:
count    702.000000
mean      29.698006
std       16.507465
min       17.000000
25%       21.000000
50%       27.000000
75%       35.000000
max      383.000000
Name: age, dtype: float64


Columna: age
DataType: float32
Unique Vals: [17. 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34.
 35. 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50. 51. 52.
 53. 54. 55. 56. nan]
Description:
count    702.000000
mean      28.860399
std        9.193460
min       17.000000
25%       21.000000
50%       27.000000
75%       34.000000
max       56.000000
Name: age, dtype: float64


#### Categorico a numérico

#### Estandarización